# Are the animals categorical in shape space? (IBL Brainwide Map)

The test run on the autism dataset (`asd_subject_gaussian.ipynb`), here on the Repeated-Site
recordings of the Brainwide Map. Every animal in this release is a wild-type control, so this
is the controls-only panel of that figure with a different, larger set of control animals.

Each animal is a point -- its whole population of neurons, compared with the Procrustes
distance -- and the question is whether those points fall into discrete groups or form one
continuous cloud.

Silhouette cannot answer that directly: it needs at least two clusters, so it can never score
the one-cluster hypothesis. That hypothesis is simulated instead, by drawing points from a
**single Gaussian** matched to the real cloud's mean and covariance -- the real spread, none of
the lumpiness. Sweep k on the data, sweep k identically on each draw, and compare the best
silhouette with the null's own best. Because the null makes the same free choice of k,
choosing k by `argmax` costs nothing.

No cross-validated splits. Splitting an animal's neurons in two exists so that an animal can be
compared with itself, which is what an identity or noise-floor question needs; here every
comparison is between two different animals, so each animal's whole population is used once.

Same test as `duszkiewicz_analyses/notebooks/hd_clustering.ipynb` panel d (head-direction
subjects) and `Posani/notebook.ipynb` section 4 (cortical regions).

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import tqdm

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))
import shapemetrics as sm                                            # noqa: E402

OUT = Path.cwd() / "results_bwm"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"svg.fonttype": "none", "text.usetex": False})

# house style of the head-direction figure: grey is always the null, dark red the data
PANEL, NULLC, DATAC = sm.PANEL, sm.NULLC, sm.DATAC
N_NULL, N_PCS = 500, 20    # 500 draws: p resolves to 1/501, at a few minutes per null
MIN_NEURONS = 100          # per animal, so an animal is a population rather than a handful


## The data

One row per neuron: its trial-averaged response over 9 signed contrasts x 10 time bins, 90
features in all, from `extract_bwm_tuning.py` (response-aligned, 0-400 ms, the five
Repeated-Site areas). Animals contributing fewer than `MIN_NEURONS` neurons would be dropped --
each animal has to stand as a population -- but in this release none are.

Subject and session are one-to-one here, one recording per animal, so "animal" and "session"
cannot be separated.

In [ ]:
z = np.load("data_bwm/bwm_tuning.npz", allow_pickle=True)
X, animal, region = z["X"].astype(float), z["subject"], z["region"]

counts = {a: int((animal == a).sum()) for a in np.unique(animal)}
keep = np.array([counts[a] >= MIN_NEURONS for a in animal])
X, animal, region = X[keep], animal[keep], region[keep]

subjects = np.unique(animal)
n_per = np.array([counts[s] for s in subjects])
print(f"{len(X)} neurons, {X.shape[1]} features, {len(subjects)} animals, "
      f"{n_per.min()}-{n_per.max()} neurons each "
      f"({int((~keep).sum())} neurons dropped with their animals)")
print(f"{len(np.unique(region))} regions: {', '.join(map(str, np.unique(region)))}")

## Subject space

Each animal's neurons become one point cloud -- its 90 features by `N_PCS` principal
components -- and animals are compared with the Procrustes distance, whole population against
whole population. The matrix is cached, so it is built once.

In [ ]:
f = OUT / "subject_procrustes.npz"
if f.exists():
    D_all = np.load(f)["D"]
else:
    D_all = sm.distance_matrix(X, animal, n_pcs=N_PCS)
    np.savez(f, D=D_all)

print(f"{D_all.shape[0]} x {D_all.shape[0]} Procrustes distances, "
      f"{D_all[np.triu_indices_from(D_all, 1)].mean():.3f} on average")


## The test

`embed` puts a distance matrix in a 5-dimensional Euclidean space, because k-means needs
coordinates. `silhouette_sweep` is Posani et al.'s statistic -- best-of-50-restarts k-means at
each k -- and `gaussian_null` repeats it on draws from one Gaussian matched to that embedding.
k runs from 2 to n - 1, and the null is given the same range.

Each null is `N_NULL` sweeps, and a sweep is 31 values of k with 50 restarts each, so this is
the slow part of the notebook: a few minutes.

In [ ]:
KS = np.arange(2, len(subjects))              # 2 .. n-1


def run(D, save_id):
    '''Data sweep and matched-Gaussian null for one set of animals, cached.

    The cache is keyed on `N_NULL` as well as on the file name, so changing the
    number of draws recomputes rather than silently reusing a null of a
    different size.
    '''
    def compute():
        E = sm.mds(D)
        return dict(obs_sweep=sm.silhouette_sweep(E, KS), ks=KS, emb=E,
                    null_sweep=sm.gaussian_null(E, KS, N_NULL,
                                                progress=lambda n: tqdm.trange(n, desc=save_id)))

    d = sm.cached_npz(OUT / f"subject_gaussian_{save_id}.npz", compute,
                      valid=lambda d: len(d["null_sweep"]) == N_NULL)
    sweep, null_sweep, ks = d["obs_sweep"], d["null_sweep"], d["ks"]

    r = sm.null_stats(sweep.max(), null_sweep.max(1))   # the null picks its own best k too
    r.update(k=int(ks[sweep.argmax()]), n=len(D),
             k2=float(np.mean(ks[null_sweep.argmax(1)] == 2)))
    print(f"{save_id:<8} n = {r['n']:>2}   silhouette {r['obs']:.4f} (k = {r['k']})   "
          f"Gaussian null {r['null'].mean():.4f} +/- {r['null'].std():.4f}   "
          f"z = {r['z']:+.2f}, p = {r['p']:.4f}   (null picks k = 2 in {r['k2']:.0%})")
    return r


res_all = run(D_all, "all")


## A second null: shuffled subjects

The Gaussian asks whether the cloud of animals is lumpy. This asks the prior question: does
animal identity structure the space at all? The statistic is the same -- best silhouette over
k -- but the null reassigns every neuron to a random animal, keeping how many neurons each
animal has, and **rebuilds everything**: the per-animal PCA, all 528 Procrustes distances, the
embedding, the sweep.

Rebuilding matters here, unlike the genotype shuffle of the autism notebook. The distance
matrix is built animal by animal, so permuting labels on a fixed matrix would compare an
animal's real geometry with itself under a new name -- close to tautological. A shuffled draw
has to be a genuinely re-derived set of pseudo-animals.

This is the expensive null in principle, but the populations here are only 90-dimensional, so
a draw costs under a second.

In [ ]:
f = OUT / "subject_shuffle.npz"
if f.exists() and len(np.load(f)["null"]) == N_NULL:
    sh_null = np.load(f)["null"]
else:
    rng = np.random.default_rng(0)
    sh_null = np.array([sm.best_silhouette(sm.mds(sm.distance_matrix(
        X, rng.permutation(animal), n_pcs=N_PCS)), KS)
        for _ in tqdm.trange(N_NULL, desc="subject shuffles")])
    np.savez(f, null=sh_null)

res_shuf = sm.null_stats(res_all["obs"], sh_null)     # same statistic, different null
res_shuf.update(n=res_all["n"], k=res_all["k"])
print(f"shuffle  n = {res_shuf['n']}   silhouette {res_shuf['obs']:.4f} (k = {res_shuf['k']})"
      f"   shuffled subjects {sh_null.mean():.4f} +/- {sh_null.std():.4f}   "
      f"z = {res_shuf['z']:+.2f}, p = {res_shuf['p']:.4f}")


## The figure

Grey is always the null, dark red always the data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(2 * PANEL, PANEL))
sm.null_panel(axes[0], res_all["obs"], res_all["null"], "best silhouette over k",
              "one continuous cloud", "real animals",
              title=f"Brainwide Map controls (n = {res_all['n']})",
              z=res_all["z"], p=res_all["p"])
sm.null_panel(axes[1], res_shuf["obs"], res_shuf["null"], "best silhouette over k",
              "shuffled subjects", "real animals",
              title=f"Brainwide Map controls (n = {res_shuf['n']})",
              z=res_shuf["z"], p=res_shuf["p"])

sm.save(fig, OUT / "bwm_subject_gaussian")
plt.show()

print(f"{N_NULL} draws per null, so the smallest attainable p is {1 / (N_NULL + 1):.4f}\n")
for name, r in [("Gaussian cloud   ", res_all), ("shuffled subjects", res_shuf)]:
    print(f"{name}  n = {r['n']}   {r['obs']:.4f} vs {r['null'].mean():.4f} "
          f"+/- {r['null'].std():.4f}   z = {r['z']:+.2f}, p = {r['p']:.4f}, k = {r['k']}")


## What it says

Same data, same statistic (best silhouette over k = 0.247, at k = 2), two nulls:

| null | what it destroys | null value | z | p |
|---|---|---|---|---|
| matched Gaussian | the lumpiness, keeping the spread | 0.270 +/- 0.035 | -0.63 | 0.73 |
| shuffled subjects | animal identity, rebuilding everything | 0.192 +/- 0.019 | **+2.92** | 0.0060 |

The two answers are not in conflict; they answer different questions.

**Animal identity is real.** Reassign the neurons to random animals and the best silhouette
falls from 0.247 to 0.192 -- an animal's own neurons make a more coherent, better-separated
population than an arbitrary set of the same size. There is genuine between-animal structure
here.

**But the space is a continuum.** That same 0.247 does not beat a single Gaussian cloud matched
to the animals' own spread (0.270). The animals are spread out and distinguishable, but they
do not fall into groups: a continuous cloud with the same covariance reproduces the observed
clustering, and then some.

An organised, continuous space is exactly the combination that shows up elsewhere in the
project -- the head-direction subjects (`hd_clustering.ipynb`: shuffle z = +3.35, Gaussian
z = +0.58) and Posani's cortical regions (hierarchy predicted from the geometry, Gaussian
z = +0.47) behave the same way. It also shows why the Gaussian null is the one that answers
"categorical or not": a shuffle can only tell you the labels mean something.

The autism release is the one exception so far, where all 36 animals reached z = +4.78 against
the Gaussian -- driven by five outlying animals rather than by a genotype split
(`asd_subject_gaussian.ipynb`).

Note also that k-means will halve any elongated cloud: the Gaussian null itself picks k = 2 in
72% of its own draws, so the data's k = 2 carries no information on its own.